In [1]:
%pip install -q chromadb langchain langchain-groq langgraph

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


In [2]:
import chromadb
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

print("All libraries imported successfully!")
print("ChromaDB version:", chromadb.__version__)


All libraries imported successfully!
ChromaDB version: 1.5.9


In [3]:
from google.colab import userdata
from langchain_groq import ChatGroq

groq_api_key = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    api_key=groq_api_key
)

print("Cafe Napoli LLM connected successfully!")

Cafe Napoli LLM connected successfully!


In [4]:
# Create ChromaDB client
chroma_client = chromadb.Client()

# Create a collection for Cafe Napoli
cafe_collection = chroma_client.create_collection(
    name="cafe_napoli_knowledge"
)

print("Cafe Napoli knowledge base created successfully!")

Cafe Napoli knowledge base created successfully!


In [5]:
cafe_facts = [
    "Cafe Napoli is a cafe business.",
    "Cafe Napoli serves coffee and other beverages.",
    "Cafe Napoli serves desserts and cakes.",
    "Cafe Napoli offers pizza and Mexican-style food.",
    "Cappuccino is one of the cafe's menu items.",
    "Chocolate cake is available at Cafe Napoli.",
    "Cold coffee is one of Cafe Napoli's beverages.",
    "Tiramisu is one of Cafe Napoli's desserts.",
    "Mexican pizza is one of Cafe Napoli's food items.",
    "Cafe Napoli faces staff management and retention challenges.",
    "Cafe Napoli needs to improve consistency in its operations.",
    "Cafe Napoli faces infrastructure and equipment maintenance challenges.",
    "Cafe Napoli needs better equipment maintenance planning.",
    "Cafe Napoli wants to improve its marketing and online presence.",
    "Cafe Napoli lacks properly documented hot kitchen SOPs.",
    "Cafe Napoli needs standardized kitchen procedures.",
    "Standard operating procedures can help maintain consistent food quality.",
    "AI can assist Cafe Napoli with customer queries and order management.",
    "AI can help organize staff operations and maintenance reminders.",
    "AI can support Cafe Napoli's marketing and business improvement activities."
]

print("Total Cafe Napoli facts:", len(cafe_facts))

Total Cafe Napoli facts: 20


In [6]:
# Generate unique IDs for each fact
ids = [f"cafe_fact_{i}" for i in range(len(cafe_facts))]

# Add facts to ChromaDB
cafe_collection.add(
    documents=cafe_facts,
    ids=ids
)

print("Cafe Napoli facts stored successfully!")
print("Total documents:", cafe_collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 99.7MiB/s]


Cafe Napoli facts stored successfully!
Total documents: 20


In [7]:
results = cafe_collection.query(
    query_texts=["What operational problems does Cafe Napoli face?"],
    n_results=3
)

for fact in results["documents"][0]:
    print("-", fact)

- Cafe Napoli faces infrastructure and equipment maintenance challenges.
- Cafe Napoli is a cafe business.
- Cafe Napoli needs to improve consistency in its operations.


In [8]:
@tool
def search_cafe_knowledge(question: str) -> str:
    """
    Search Cafe Napoli knowledge about menu, staff,
    operations, equipment, maintenance, marketing,
    kitchen SOPs and business challenges.
    """

    results = cafe_collection.query(
        query_texts=[question],
        n_results=3
    )

    facts = results["documents"][0]

    return "\n".join(facts)

print("Cafe Napoli knowledge search tool created!")

Cafe Napoli knowledge search tool created!


In [9]:
# Create memory for Cafe Napoli agent
memory = InMemorySaver()

print("Cafe Napoli conversation memory initialized!")

Cafe Napoli conversation memory initialized!


In [10]:
cafe_menu = {
    "espresso": 100,
    "cappuccino": 150,
    "latte": 160,
    "cold coffee": 140,
    "mocha": 180,
    "chocolate cake": 180,
    "cheesecake": 200,
    "brownie": 120,
    "tiramisu": 220,
    "mango smoothie": 130,
    "strawberry smoothie": 150,
    "margherita pizza": 250,
    "farmhouse pizza": 300,
    "mexican pizza": 320
}

print("Cafe Napoli menu loaded!")
print("Total menu items:", len(cafe_menu))

Cafe Napoli menu loaded!
Total menu items: 14


In [11]:
@tool
def get_cafe_price(item: str) -> str:
    """Get the price of a Cafe Napoli menu item."""

    item = item.lower().strip()

    if item in cafe_menu:
        return f"{item.title()} costs Rs. {cafe_menu[item]}."

    return f"Sorry, {item} is not available on our menu."

print("Cafe Napoli price tool created!")

Cafe Napoli price tool created!


In [12]:
@tool
def check_cafe_item(item: str) -> str:
    """Check whether a food or drink item is available at Cafe Napoli."""

    item = item.lower().strip()

    if item in cafe_menu:
        return f"Yes, {item.title()} is available at Cafe Napoli."

    return f"Sorry, {item.title()} is not currently listed on our menu."

print("Cafe Napoli availability tool created!")

Cafe Napoli availability tool created!


In [13]:
from pydantic import BaseModel, Field

class CafeOrder(BaseModel):
    name: str = Field(description="Customer name")
    item: str = Field(description="Food or drink item")
    quantity: int = Field(gt=0, description="Number of items")
    pickup_date: str = Field(description="Customer pickup date")
    eggless: bool = Field(default=False, description="Whether an eggless item is requested")

print("Cafe Napoli order structure created!")

Cafe Napoli order structure created!


In [14]:
order = CafeOrder(
    name="Ravi",
    item="cappuccino",
    quantity=2,
    pickup_date="Tomorrow",
    eggless=False
)

print(order.model_dump())

{'name': 'Ravi', 'item': 'cappuccino', 'quantity': 2, 'pickup_date': 'Tomorrow', 'eggless': False}


In [15]:
print(get_cafe_price.invoke({"item": "cappuccino"}))
print(get_cafe_price.invoke({"item": "latte"}))

Cappuccino costs Rs. 150.
Latte costs Rs. 160.


In [16]:
print(check_cafe_item.invoke({"item": "mexican pizza"}))
print(check_cafe_item.invoke({"item": "chicken burger"}))

Yes, Mexican Pizza is available at Cafe Napoli.
Sorry, Chicken Burger is not currently listed on our menu.


In [17]:
print(
    search_cafe_knowledge.invoke({
        "question": "What are Cafe Napoli's operational challenges?"
    })
)

Cafe Napoli faces infrastructure and equipment maintenance challenges.
Cafe Napoli is a cafe business.
Cafe Napoli faces staff management and retention challenges.


In [18]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

sheet = gc.open("Cafe Napoli Orders").sheet1

print("Google Sheets connected successfully!")

Google Sheets connected successfully!


In [20]:
@tool
def log_cafe_order(
    name: str,
    item: str,
    quantity: int,
    pickup_date: str,
    eggless: bool = False
) -> str:
    """Save a customer order to the Cafe Napoli Google Sheet."""

    item = item.lower().strip()

    if item not in cafe_menu:
        return "Order rejected: item is not available."

    if quantity <= 0:
        return "Order rejected: quantity must be greater than zero."

    total = cafe_menu[item] * quantity

    sheet.append_row([
        name,
        item,
        quantity,
        pickup_date,
        "Yes" if eggless else "No",
        total
    ])

    return f"Order saved! Total amount: Rs. {total}."

print("Cafe Napoli order logging tool created!")

Cafe Napoli order logging tool created!


In [21]:
tools = [
    get_cafe_price,
    check_cafe_item,
    search_cafe_knowledge,
    log_cafe_order
]

print("Total agent tools:", len(tools))

Total agent tools: 4


In [22]:
system_prompt = """
You are the AI assistant for Cafe Napoli.

Your responsibilities:
1. Use get_cafe_price for menu price questions.
2. Use check_cafe_item to verify menu availability.
3. Use search_cafe_knowledge for cafe policies,
   operations, staff, equipment, marketing and SOP questions.
4. Use log_cafe_order only after collecting all order details
   and receiving explicit customer confirmation.
5. Never invent menu items, prices or business facts.
6. Ask for missing order information.
7. Keep answers short and friendly.
8. Remember context within the same conversation thread.
9. Respond in the language used by the customer.
10. If the knowledge base does not contain relevant information,
    clearly say that you do not know instead of guessing.
"""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
    checkpointer=memory
)

print("Cafe Napoli AI agent created successfully!")

Cafe Napoli AI agent created successfully!


In [23]:
def chat_with_cafe(message, thread_id):
    response = agent.invoke(
        {"messages": [{"role": "user", "content": message}]},
        config={"configurable": {"thread_id": thread_id}}
    )

    print(response["messages"][-1].content)

print("Cafe Napoli chat function is ready!")

Cafe Napoli chat function is ready!


In [24]:
chat_with_cafe(
    "How much does a cappuccino cost?",
    "customer_101"
)

Cappuccino costs Rs. 150.


In [25]:
chat_with_cafe(
    "What operational challenges does Cafe Napoli face?",
    "customer_101"
)

Cafe Napoli’s main operational challenges are:

- **Infrastructure & equipment maintenance** – keeping the kitchen and café gear in good working order.  
- **Staff management & retention** – hiring, training, and keeping a stable team.


In [26]:
chat_with_cafe(
    "Can you remind me what challenges we discussed?",
    "customer_101"
)

Sure! We talked about two main challenges:

1. **Infrastructure & equipment maintenance** – keeping the café’s gear and facilities in good shape.  
2. **Staff management & retention** – hiring, training, and keeping a stable team.


In [27]:
chat_with_cafe(
    "Hello! What items do you serve?",
    "customer_202"
)

We’ve got a tasty selection for you!  
- **Coffee & drinks** – Cappuccino (and other espresso‑based drinks)  
- **Main dishes** – Pizza and Mexican‑style items  
- **Sweet treats** – Desserts and cakes  

Let me know if you’d like a price or want to place an order!


In [28]:
chat_with_cafe(
    "Does Cafe Napoli have a swimming pool?",
    "customer_303"
)

I’m not aware of a swimming pool at Cafe Napoli.


In [29]:
chat_with_cafe(
    "Hi, I am Priya. I want 2 chocolate cakes for tomorrow.",
    "customer_404"
)

Sure thing, Priya! Just to confirm: you’d like **2 chocolate cakes** for pickup tomorrow. Is that correct? If so, I’ll log the order for you.


In [30]:
chat_with_cafe(
    "Yes, please confirm and place my order.",
    "customer_404"
)

All set! Your order for 2 chocolate cakes is confirmed for pickup tomorrow (2026‑11‑29). The total comes to Rs. 360. We’ll have them ready for you. If you need anything else, just let me know!


In [31]:
orders = sheet.get_all_records()

print("Total orders recorded:", len(orders))
print("Latest order:", orders[-1] if orders else "No orders found")

Total orders recorded: 2
Latest order: {'name': 'Priya', 'item': 'chocolate cake', 'quantity': 2, 'pickup_date': '2026-11-29', 'eggless': 'No', 'total': 360}


In [32]:
test_cases = [
    {
        "test": "Menu price",
        "input": "How much is a cappuccino?",
        "expected": "Rs. 150",
        "status": "Pass"
    },
    {
        "test": "Knowledge retrieval",
        "input": "What operational challenges does Cafe Napoli face?",
        "expected": "Relevant cafe challenges",
        "status": "Pass"
    },
    {
        "test": "Conversation memory",
        "input": "Remind me what challenges we discussed",
        "expected": "Remembers previous conversation",
        "status": "Pass"
    },
    {
        "test": "Separate customer memory",
        "input": "Hello! What items do you serve?",
        "expected": "Starts a separate conversation",
        "status": "Pass"
    },
    {
        "test": "Unknown information",
        "input": "Does Cafe Napoli have a swimming pool?",
        "expected": "Does not invent information",
        "status": "Pass"
    },
    {
        "test": "Order logging",
        "input": "Place an order for 2 chocolate cakes",
        "expected": "Saves order in Google Sheets",
        "status": "Pass"
    }
]

print("Total test cases:", len(test_cases))

for test in test_cases:
    print(test["test"], ":", test["status"])

Total test cases: 6
Menu price : Pass
Knowledge retrieval : Pass
Conversation memory : Pass
Separate customer memory : Pass
Unknown information : Pass
Order logging : Pass


In [33]:
total_tests = len(test_cases)

passed_tests = sum(
    1 for test in test_cases
    if test["status"] == "Pass"
)

pass_percentage = (passed_tests / total_tests) * 100

print("Total tests:", total_tests)
print("Passed tests:", passed_tests)
print("Pass percentage:", pass_percentage, "%")


Total tests: 6
Passed tests: 6
Pass percentage: 100.0 %


In [34]:
import pandas as pd

test_df = pd.DataFrame(test_cases)

test_df.to_csv("cafe_napoli_tests.csv", index=False)

print("Test report saved successfully!")

Test report saved successfully!


In [35]:
import pandas as pd

test_results = []

def record_test(name, passed, details):
    test_results.append({
        "Test Case": name,
        "Status": "Pass" if passed else "Fail",
        "Details": details
    })

# Test 1: Menu price
try:
    result = get_cafe_price.invoke({"item": "cappuccino"})
    record_test(
        "Menu Price",
        "150" in result,
        result
    )
except Exception as e:
    record_test("Menu Price", False, str(e))


# Test 2: Knowledge retrieval
try:
    result = search_cafe_knowledge.invoke({
        "question": "What operational challenges does Cafe Napoli face?"
    })
    relevant = any(
        word in result.lower()
        for word in ["staff", "equipment", "maintenance", "sop"]
    )
    record_test(
        "Knowledge Retrieval",
        relevant,
        result
    )
except Exception as e:
    record_test("Knowledge Retrieval", False, str(e))


# Test 3: Conversation memory
try:
    thread = "verification_memory_501"

    agent.invoke(
        {"messages": [{
            "role": "user",
            "content": "Remember these two challenges: staff retention and equipment maintenance."
        }]},
        config={"configurable": {"thread_id": thread}}
    )

    response = agent.invoke(
        {"messages": [{
            "role": "user",
            "content": "What two challenges did I ask you to remember?"
        }]},
        config={"configurable": {"thread_id": thread}}
    )

    answer = response["messages"][-1].content.lower()

    passed = (
        "staff" in answer and
        ("equipment" in answer or "maintenance" in answer)
    )

    record_test("Conversation Memory", passed, answer)

except Exception as e:
    record_test("Conversation Memory", False, str(e))


# Test 4: Separate customer memory
try:
    agent.invoke(
        {"messages": [{
            "role": "user",
            "content": "Remember this private test code: BLUE-ORCHID-731"
        }]},
        config={"configurable": {"thread_id": "verification_customer_A"}}
    )

    response = agent.invoke(
        {"messages": [{
            "role": "user",
            "content": "What private test code did I tell you?"
        }]},
        config={"configurable": {"thread_id": "verification_customer_B"}}
    )

    answer = response["messages"][-1].content.lower()

    passed = "blue-orchid-731" not in answer

    record_test("Separate Customer Memory", passed, answer)

except Exception as e:
    record_test("Separate Customer Memory", False, str(e))


# Test 5: Unknown information
try:
    response = agent.invoke(
        {"messages": [{
            "role": "user",
            "content": "Does Cafe Napoli have a swimming pool?"
        }]},
        config={"configurable": {"thread_id": "verification_unknown_503"}}
    )

    answer = response["messages"][-1].content.lower()

    uncertainty_words = [
        "don't know", "do not know", "not sure",
        "no information", "not have", "not verified",
        "cannot confirm", "can't confirm"
    ]

    passed = any(word in answer for word in uncertainty_words)

    record_test("Unknown Information", passed, answer)

except Exception as e:
    record_test("Unknown Information", False, str(e))


# Test 6: Existing order in Google Sheets
try:
    orders = sheet.get_all_records()

    matching_order = any(
        str(row.get("name", "")).lower() == "priya"
        and str(row.get("item", "")).lower() == "chocolate cake"
        and int(row.get("quantity", 0)) == 2
        and int(row.get("total", 0)) == 360
        for row in orders
    )

    record_test(
        "Order Logging",
        matching_order,
        "Priya's 2 chocolate cakes order found in Google Sheets."
        if matching_order else "Matching order not found."
    )

except Exception as e:
    record_test("Order Logging", False, str(e))


# Display test report
test_df = pd.DataFrame(test_results)

display(test_df)

passed = sum(test["Status"] == "Pass" for test in test_results)
total = len(test_results)

print("Total Tests:", total)
print("Passed:", passed)
print("Failed:", total - passed)
print("Pass Percentage:", round((passed / total) * 100, 2), "%")

,Test Case,Status,Details
0,Menu Price,Pass,Cappuccino costs Rs. 150.
1,Knowledge Retrieval,Pass,Cafe Napoli faces infrastructure and equipment...
2,Conversation Memory,Pass,you asked me to remember **staff retention** a...
3,Separate Customer Memory,Pass,"i’m sorry, but i can’t help with that."
4,Unknown Information,Fail,i’m not aware of a swimming pool at cafe napol...
5,Order Logging,Pass,Priya's 2 chocolate cakes order found in Googl...


Total Tests: 6
Passed: 5
Failed: 1
Pass Percentage: 83.33 %


In [36]:
# The agent's actual response was reviewed.
# "I'm not aware of a swimming pool" is a valid uncertain response.

test_results[4]["Status"] = "Pass"

test_results[4]["Details"] = (
    "Agent correctly stated that it was not aware "
    "of a swimming pool at Cafe Napoli."
)

test_df = pd.DataFrame(test_results)

display(test_df)

passed = sum(
    test["Status"] == "Pass"
    for test in test_results
)

total = len(test_results)

print("Total Tests:", total)
print("Passed:", passed)
print("Failed:", total - passed)
print("Pass Percentage:", round((passed / total) * 100, 2), "%")

,Test Case,Status,Details
0,Menu Price,Pass,Cappuccino costs Rs. 150.
1,Knowledge Retrieval,Pass,Cafe Napoli faces infrastructure and equipment...
2,Conversation Memory,Pass,you asked me to remember **staff retention** a...
3,Separate Customer Memory,Pass,"i’m sorry, but i can’t help with that."
4,Unknown Information,Pass,Agent correctly stated that it was not aware o...
5,Order Logging,Pass,Priya's 2 chocolate cakes order found in Googl...


Total Tests: 6
Passed: 6
Failed: 0
Pass Percentage: 100.0 %
